In [1]:
fiber_files = [f'sox2_fibers/fiber_sox2_{i}_{j}.h5' for j in range(1,4) for i in (19,39,50,'removed')]
traj_files = [f'MC_1_trajectories/sox2_traj_{i}_{j}.h5' for j in range(1,4) for i in (19,39,50,'removed')]

In [129]:
with open('run.sh','r') as f:
    bp = f.readlines()
run_line = bp[-1].split()
for i in (19,39,50,'removed'):
    file_text = bp[:-1].copy()
    for j in range(1,4):
        f_file = f'sox2_fibers/fiber_sox2_{i}_{j}.h5'
        t_file = f'MC_2_trajectories/sox2_traj_{i}_{j}.h5'
        o_file = f'MC_2_logs/log_sox2_{i}_{j}.txt'
        run_line_j = run_line.copy()
        run_line_j.insert(-3,o_file)
        run_line_j.insert(-2,t_file)
        run_line_j.insert(-1,f_file)
        file_text.append(' '.join(run_line_j)+'\n')
    file_text.append('wait')
    with open(f'MC_2_run_files/run_sox2_{i}.sh','w') as f:
        f.write(''.join(file_text))

In [24]:
with open('test.txt','a') as f:
    f.write('123')

In [130]:
for i in (19,39,50,'removed'):
    !sbatch MC_2_run_files/run_sox2_{i}.sh

Submitted batch job 22124
Submitted batch job 22125
Submitted batch job 22126
Submitted batch job 22127


In [121]:
for i in range(4):
    !sbatch MC_2_run_files/run_sox2_{19}.sh

Submitted batch job 22118
Submitted batch job 22119
Submitted batch job 22120
Submitted batch job 22121


In [126]:
for i in range(22118,22122):
    !scancel {i}

In [134]:
!squeue

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
             21704       gpu linker30 afedulov  R 8-22:08:38      1 node08
             21705       gpu linker30 afedulov  R 8-22:06:48      1 node08
             21992       gpu prechrom afedulov  R 2-00:13:39      1 node09
             21993       gpu prechrom afedulov  R 2-00:13:28      1 node09
             22124       gpu CG_fiber vvasilev  R       3:39      1 node10
             22125       gpu CG_fiber vvasilev  R       3:36      1 node10
             22126       gpu CG_fiber vvasilev  R       3:36      1 node02
             22127       gpu CG_fiber vvasilev  R       3:36      1 node02
             21837    memory interact guowenxi  R 3-01:56:49      1 node12
             21838    memory interact guowenxi  R 3-01:54:17      1 node12
             22038    memory interact guowenxi  R   12:49:46      1 node12
             22040    memory interact guowenxi  R   12:43:00      1 node12
             21

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import pynamod

In [2]:
import h5py

In [6]:
cgs = pynamod.CG_Structure()
file = h5py.File(fiber_files[0],'r')
cgs.load_from_h5(file)
cgs.dna.transfer_trajectory_to_h5(traj_files[0],'r')

/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/pynamod/structures/DNA_structure.py:264: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  self.movable_steps = torch.tensor(data['movable_steps'])
/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/pynamod/structures/DNA_structure.py:264: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  self.movable_steps = torch.tensor(data['movable_steps'])
/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/pynamod/structures/DNA_structure.py:264: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  self.movable_steps = torch.tensor(data['movable_steps'])
/home/vvasilev/.conda/envs/pynamod/lib/python3.11/site-packages/pynamod/structures/DNA_structure.py:264: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted

In [7]:
cgs.dna.trajectory.traj_step = 1000

In [37]:
u = cgs.get_cg_mda_traj()

In [9]:
import nglview as nv

In [38]:
nv.show_mdanalysis(u)

NGLWidget(max_frame=300)

In [4]:
import numpy as np
import pandas as pd
from tqdm.std import tqdm

In [9]:
for n in traj_files[1:]:
    f = h5py.File(n,'r')
    keys = sorted(f.keys())
    ens = np.array(
            [f[k]["energies"][:3] for k in tqdm(keys)],
            dtype=f[keys[0]]["energies"].dtype,
        )
    
    df = pd.DataFrame(data = ens, columns=['bend','elst','ld']).to_csv('energies/'+n[18:-3]+'.csv')
    f.close()

    del df
    del ens
    del keys

 24%|██▍       | 71699/300001 [10:08<31:05, 122.36it/s] 

KeyboardInterrupt: 

 24%|██▍       | 71699/300001 [10:25<31:05, 122.36it/s]